# Step 01 — Project Initialization (EU-27)

Reproducible setup for the AI Realization Gap project. This version matches the supplied Step 01 outputs and prepares the exact master inputs used by Steps 02 and 03. Runtime-specific metadata fields (timestamp, Python version and platform) naturally change between runs.

In [ ]:
from pathlib import Path
import json
import platform
import sys
from datetime import datetime, timezone
import pandas as pd

# Google Colab: mount Drive; local Python: use the local fallback folder.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/AI_Realization_Gap_EU27')
except ImportError:
    PROJECT_ROOT = Path.cwd() / 'AI_Realization_Gap_EU27'

FOLDERS = ['00_admin', '01_raw_data', '02_intermediate_data',
           '03_processed_data', '04_code', '05_tables', '06_figures',
           '07_models', '08_logs', '09_manuscript']
for folder in FOLDERS:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
EU27 = {
    'AT': ('Austria','AUT'), 'BE': ('Belgium','BEL'),
    'BG': ('Bulgaria','BGR'), 'HR': ('Croatia','HRV'),
    'CY': ('Cyprus','CYP'), 'CZ': ('Czechia','CZE'),
    'DK': ('Denmark','DNK'), 'EE': ('Estonia','EST'),
    'FI': ('Finland','FIN'), 'FR': ('France','FRA'),
    'DE': ('Germany','DEU'), 'EL': ('Greece','GRC'),
    'HU': ('Hungary','HUN'), 'IE': ('Ireland','IRL'),
    'IT': ('Italy','ITA'), 'LV': ('Latvia','LVA'),
    'LT': ('Lithuania','LTU'), 'LU': ('Luxembourg','LUX'),
    'MT': ('Malta','MLT'), 'NL': ('Netherlands','NLD'),
    'PL': ('Poland','POL'), 'PT': ('Portugal','PRT'),
    'RO': ('Romania','ROU'), 'SK': ('Slovakia','SVK'),
    'SI': ('Slovenia','SVN'), 'ES': ('Spain','ESP'),
    'SE': ('Sweden','SWE')
}
country_master = pd.DataFrame(
    [{'geo': geo, 'country': name, 'iso3': iso3}
     for geo, (name, iso3) in EU27.items()]
)
YEARS = [2021, 2023, 2024, 2025]
panel = pd.DataFrame(
    [{**r, 'year': y} for r in country_master.to_dict('records') for y in YEARS],
    columns=['geo', 'country', 'iso3', 'year']
).sort_values(['iso3', 'year']).reset_index(drop=True)
print('Countries:', len(country_master), '| Years:', YEARS, '| Panel rows:', len(panel))
display(panel.head(8))

In [ ]:
qc = pd.DataFrame([
    ('EU27 has exactly 27 countries', len(country_master) == 27),
    ('All geo codes unique', country_master['geo'].is_unique),
    ('All ISO3 codes unique', country_master['iso3'].is_unique),
    ('Core AI years correct', sorted(panel['year'].unique().tolist()) == YEARS),
    ('Panel contains 108 observations', len(panel) == 108),
    ('No missing geo values', panel['geo'].notna().all()),
    ('No missing country values', panel['country'].notna().all()),
    ('No missing ISO3 values', panel['iso3'].notna().all()),
    ('No duplicated country-year', ~panel.duplicated(['geo', 'year']).any())
], columns=['check', 'passed'])
assert qc['passed'].all(), qc.loc[~qc['passed']].to_string(index=False)
print(qc.to_string(index=False))

In [ ]:
metadata = {
    'project_name': 'AI Realization Gap EU27',
    'working_title': 'From AI Potential to AI Adoption: Explaining the AI Realization Gap across EU Countries',
    'geographic_scope': 'European Union — EU-27',
    'n_countries': 27,
    'core_ai_years': YEARS,
    'maximum_country_year_observations': len(panel),
    'core_outcome': 'AI realization gap',
    'realized_ai_measure': 'Enterprise use of artificial intelligence technologies',
    'primary_realized_ai_source': 'Eurostat',
    'project_initialized': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'platform': platform.platform()
}
country_master.to_csv(PROJECT_ROOT/'00_admin'/'eu27_country_master.csv', index=False)
panel.to_csv(PROJECT_ROOT/'02_intermediate_data'/'eu27_ai_year_panel_skeleton.csv', index=False)
qc.to_csv(PROJECT_ROOT/'08_logs'/'Step_01_QC.csv', index=False)
with open(PROJECT_ROOT/'00_admin'/'project_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print('Saved:')
for f in [PROJECT_ROOT/'00_admin'/'eu27_country_master.csv',
          PROJECT_ROOT/'02_intermediate_data'/'eu27_ai_year_panel_skeleton.csv',
          PROJECT_ROOT/'08_logs'/'Step_01_QC.csv',
          PROJECT_ROOT/'00_admin'/'project_metadata.json']:
    print(' -', f)